# ESM2 Masked Language Model

![ESM2 Masked Language Model](https://proto-bio.github.io/proto-assets/images/tool/esm2/hero.png)

This notebook demonstrates the three core operations of the ESM2 tool: extracting sequence embeddings, sampling mutations, and scoring sequences. ESM2 is a protein language model trained on millions of protein sequences using masked language modeling, and its learned representations capture structural and evolutionary relationships that support a range of downstream tasks.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("esm2")
display_overview("esm2")
display_docs_section("esm2", "Background")

# ESM2

ESM2 (Evolutionary Scale Modeling 2) is Meta AI's [protein language model](https://www.evolutionaryscale.ai/blog/esm-cambrian) trained on millions of protein sequences from [UniRef](https://www.uniprot.org/help/uniref). It provides sequence embeddings, per-position amino acid logits, sequence mutation (sampling), and sequence scoring (MLM pseudo-perplexity). ESM2 offers multiple model sizes from 8M to 15B parameters, balancing quality and computational cost.

This package also includes `esm2-gradient`, a relaxed-sequence gradient tool that computes the masked pseudo-log-likelihood objective over a continuous L×20 logits distribution and returns its gradient with respect to the input. It can be used as a differentiable structure-free pLM loss inside MCMC, gradient descent, or any optimization loop over relaxed protein sequences.

## Background

**What are protein language models?**
Protein language models (pLMs) learn the "grammar" of proteins from evolutionary data. They capture:
- **Sequence conservation**: Which residues are essential for function
- **[Co-evolution](https://en.wikipedia.org/wiki/Coevolution)**: Pairs of residues that evolve together (often in contact)
- **Structural constraints**: Patterns that define [secondary/tertiary structure](https://en.wikipedia.org/wiki/Protein_structure)
- **Functional motifs**: Binding sites, active sites, [post-translational modifications](https://en.wikipedia.org/wiki/Post-translational_modification)

**Why embeddings are useful:**
ESM2 embeddings encode rich biological information:
- Similar proteins cluster in embedding space
- Embeddings predict structure, function, and localization
- Mean-pooled embeddings work as fixed-length sequence descriptors

**Why logits are useful:**
Per-position logits indicate the model's confidence in each amino acid:
- High logits = evolutionarily preferred (conserved)
- Low logits = tolerated or deleterious positions
- Comparing wild-type vs mutant logits predicts variant effects
- Logits are returned over 20 canonical amino acids in fixed order: `ACDEFGHIKLMNPQRSTVWY`

**Sequence length limit:**
ESM-2's positional encoding caps inputs at 1022 residues. Longer inputs raise a `ValueError` rather than being silently truncated; chunk longer proteins yourself before invoking any ESM-2 tool. The same 1022 cap applies to the `L` dimension of `esm2-gradient` input logits.

## Available tools

In [2]:
display_available_tools("esm2")

- **`run_esm2_embeddings()`** — Extract protein sequence embeddings and logits using ESM2
- **`run_esm2_gradient()`** — Compute ESM2 masked pseudo-log-likelihood gradient for relaxed protein sequences
- **`run_esm2_sample()`** — Sample masked positions in protein sequences using ESM2 language model
- **`run_esm2_score()`** — Score protein sequences using ESM2 language model

### `run_esm2_embeddings`

ESM2 produces dense vector representations for each input protein sequence. These mean-pooled embeddings compress the full sequence-length representation into a single fixed-dimension vector that captures structural, functional, and evolutionary properties. Embeddings from different sequences can be compared directly using cosine similarity or Euclidean distance, making them useful as features for machine learning models, for clustering related proteins, or for building sequence search indices.

In [3]:
from proto_tools import (
    ESM2EmbeddingsConfig,
    ESM2EmbeddingsInput,
    run_esm2_embeddings,
)

In [4]:
# Display docs
display_api_reference("esm2", "input", "run_esm2_embeddings")

# Two hemoglobin-like sequences to embed
sequences = [
    "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSHGSAQVKGHG",
    "MNLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSHGSAQVKGHG",
]
inputs = ESM2EmbeddingsInput(sequences=sequences)

**Input** — `ESM2EmbeddingsInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `sequences` | `list[str]` | required | Protein sequence(s) to process as string or list of strings. (will be normalized to List[str]) |

In [5]:
# Display config docs
display_api_reference("esm2", "config", "run_esm2_embeddings")

# Configure the 650M model | see docs above for all fields
config = ESM2EmbeddingsConfig(
    model_checkpoint="esm2_t33_650M_UR50D",
    batch_size=2,
    return_logits=False,
    device="cuda",  # Change to "cpu" if no GPU available
)

**Config** — `ESM2EmbeddingsConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `verbose` | `int` | `0` | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| `device` | `str` | `'cuda'` | Device to run the model on |
| `timeout` | `int` | `600` | Maximum execution time in seconds |
| `seed` | `int \| None` | `None` | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| `batch_size` | `int` | `1` | Sequences per GPU forward pass; raise for throughput, lower if OOM |
| `model_checkpoint` | `Literal['esm2_t6_8M_UR50D', 'esm2_t12_35M_UR50D', 'esm2_t30_150M_UR50D', 'esm2_t33_650M_UR50D', 'esm2_t36_3B_UR50D', 'esm2_t48_15B_UR50D']` | `'esm2_t33_650M_UR50D'` | ESM2 weights variant; trade off speed vs embedding quality |
| `return_logits` | `bool` | `False` | Include per-position logits in the output (large; disable to save memory) |
| `repr_layer` | `int` | `-1` | Transformer layer index for embeddings (0=embedding output, N=layer N, -1=last) |

In [6]:
# Run the embeddings function
embeddings_result = run_esm2_embeddings(inputs, config)

Running run_esm2_embeddings [00:00]

In [7]:
# Display output docs
display_api_reference("esm2", "output", "run_esm2_embeddings")

# Inspect the embedding dimension and first few values for each sequence
for i, emb in enumerate(embeddings_result.results):
    print(f"Sequence {i + 1}: embedding length = {len(emb.mean_embedding)}, first 5 values = {emb.mean_embedding[:5]}")

**Output** — `ESM2EmbeddingsOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `results` | `list[proto_tools.tools.masked_models.shared_data_models.SequenceEmbedding]` | required | Per-sequence embedding results |

Sequence 1: embedding length = 1280, first 5 values = [-0.010943129658699036, -0.07043585181236267, -0.04228709638118744, 0.03445253521203995, -0.08040348440408707]
Sequence 2: embedding length = 1280, first 5 values = [-0.013179144822061062, -0.06401937454938889, -0.04112853854894638, 0.034877482801675797, -0.07414675503969193]


### `run_esm2_sample`

ESM2 sampling uses the model's predicted amino acid distributions to propose mutations at masked positions. There are two ways to specify which positions to sample:

1. **Custom masks** — directly mark positions with `_` in the input sequence to control exactly which residues are redesigned
2. **Masking strategy** — let the `MaskingStrategy` config automatically select positions using random, entropy-based, or max-logit methods

#### Approach 1: Custom masks with `_`

Use the `_` character to explicitly mark which positions should be redesigned. All other positions are held fixed. This is useful when you know exactly which residues you want to mutate — for example, redesigning a loop region or a specific binding interface.

In [8]:
from proto_tools import (
    ESM2SampleConfig,
    ESM2SampleInput,
    run_esm2_sample,
)

# Display input docs
display_api_reference("esm2", "input", "run_esm2_sample")

# Mask positions 6-10 with _ to redesign that region
masked_input = ESM2SampleInput(sequences=["MVLSP_____VKAAW"])

**Input** — `ESM2SampleInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `sequences` | `list[str]` | required | Protein sequence(s) to process as string or list of strings. (will be normalized to List[str]) |

In [9]:
# Display config docs
display_api_reference("esm2", "config", "run_esm2_sample")

# No masking strategy needed — positions are already specified by _
config = ESM2SampleConfig(
    device="cuda",  # Change to "cpu" if no GPU available
)

**Config** — `ESM2SampleConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `verbose` | `int` | `0` | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| `device` | `str` | `'cuda'` | Device to run the model on |
| `timeout` | `int` | `600` | Maximum execution time in seconds |
| `seed` | `int \| None` | `None` | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| `batch_size` | `int` | `1` | Sequences per GPU forward pass; raise for throughput, lower if OOM |
| `masking_strategy` | `proto_tools.transforms.masking.base.MaskingStrategy` | `MaskingStrategy(method='random', num_mutations=None, mask_fraction=None, fixed_positions=None, temperature=1.0)` | Strategy for selecting positions to mask for resampling |
| `model_checkpoint` | `Literal['esm2_t6_8M_UR50D', 'esm2_t12_35M_UR50D', 'esm2_t30_150M_UR50D', 'esm2_t33_650M_UR50D', 'esm2_t36_3B_UR50D', 'esm2_t48_15B_UR50D']` | `'esm2_t33_650M_UR50D'` | ESM2 weights variant; trade off speed vs sample quality |
| `sampling_method` | `Literal['single_pass', 'iterative_refinement']` | `'single_pass'` | 'single_pass' samples every mask in one forward; 'iterative_refinement' runs the MaskGIT-style loop |
| `temperature` | `float` | `1.0` | Softmax temperature for per-position amino-acid sampling |
| `top_p` | `float` | `1.0` | Nucleus sampling threshold; 1.0 disables |
| `num_steps` | `int` | `20` | Iterative-refinement decoding steps; diminishing returns above 20 |
| `schedule` | `Literal['cosine', 'linear']` | `'cosine'` | Unmask schedule across rounds; 'cosine' fronts more commits late |
| `strategy` | `Literal['random', 'entropy']` | `'random'` | Position-selection per round; 'entropy' commits the most-confident first |
| `temperature_annealing` | `bool` | `True` | Anneal temperature toward 0 across rounds |
| `return_logits` | `bool` | `False` | Include per-position logits in the output (large; disable to save memory) |

In [10]:
# Run sampling on the custom-masked input
custom_mask_result = run_esm2_sample(masked_input, config)

In [11]:
# Display output docs
display_api_reference("esm2", "output", "run_esm2_sample")

# Show the original vs sampled — only the masked positions change
original = "MVLSPADKTNVKAAW"
for i, seq in enumerate(custom_mask_result.sequences):
    diffs = [j + 1 for j, (a, b) in enumerate(zip(original, seq)) if a != b]
    print(f"Original: {original}")
    print(f"Sampled:  {seq}  (mutated positions: {diffs})")

**Output** — `ESM2SampleOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `sequences` | `list[str]` | required | Sampled/restored protein sequences |
| `logits` | `list[list[list[float]]] \| None` | `None` | Per-position amino acid logits. Shape: [num_sequences, seq_len, 20]. |

Original: MVLSPADKTNVKAAW
Sampled:  MVLSPVQPLGVKAAW  (mutated positions: [6, 7, 8, 9, 10])


#### Approach 2: Automatic position selection with `MaskingStrategy`

Instead of manually placing `_` characters, you can let `MaskingStrategy` choose which positions to mask. The `method` parameter controls the selection logic:

- **`"random"`** (default) — uniform random selection
- **`"entropy"`** — targets positions where ESM2 is least certain about the original residue
- **`"max-logit"`** — targets positions where ESM2 most confidently predicts a different amino acid

Use `num_mutations` for an exact count or `mask_fraction` for a proportion of positions.

In [12]:
from proto_tools.transforms.masking import MaskingStrategy

# Entropy-based masking to target the 5 most uncertain positions
strategy_input = ESM2SampleInput(sequences=["MVLSPADKTNVKAAW"])
strategy_config = ESM2SampleConfig(
    masking_strategy=MaskingStrategy(method="entropy", num_mutations=5),
    device="cuda",  # Change to "cpu" if no GPU available
)

strategy_result = run_esm2_sample(strategy_input, strategy_config)

In [13]:
# The model automatically selected the 5 highest-entropy positions to mutate
original = "MVLSPADKTNVKAAW"
for i, seq in enumerate(strategy_result.sequences):
    diffs = [j + 1 for j, (a, b) in enumerate(zip(original, seq)) if a != b]
    print(f"Original: {original}")
    print(f"Sampled:  {seq}  (mutated positions: {diffs})")

Original: MVLSPADKTNVKAAW
Sampled:  MQLSPADKGNNLAAW  (mutated positions: [2, 9, 11, 12])


### `run_esm2_score`

Pseudo-perplexity scoring masks each position in a sequence one at a time and measures how well the model predicts the original residue. Lower perplexity indicates that the sequence is more consistent with the evolutionary patterns the model learned during training, while higher perplexity suggests the sequence contains unusual or unlikely amino acid choices. This is useful for ranking designed sequences, filtering out poor candidates before expensive experimental validation, or identifying anomalous regions within a protein.

In [14]:
from proto_tools import (
    ESM2ScoringConfig,
    ESM2ScoringInput,
    run_esm2_score,
)

In [15]:
# Display docs
display_api_reference("esm2", "input", "run_esm2_score")

# Two short sequences to score
inputs = ESM2ScoringInput(sequences=["MKTAYIAKQR", "EVQLVESGGS"])

**Input** — `ESM2ScoringInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `sequences` | `list[str]` | required | Protein sequence(s) to process as string or list of strings. (will be normalized to List[str]) |

In [16]:
# Display config docs
display_api_reference("esm2", "config", "run_esm2_score")

# Process 5 masked variants per forward pass | see docs above for all fields
config = ESM2ScoringConfig(
    batch_size=5,
    return_logits=False,
    device="cuda",  # Change to "cpu" if no GPU available
)

**Config** — `ESM2ScoringConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `verbose` | `int` | `0` | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| `device` | `str` | `'cuda'` | Device to run the model on |
| `timeout` | `int` | `600` | Maximum execution time in seconds |
| `seed` | `int \| None` | `None` | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| `batch_size` | `int` | `1` | Sequences per GPU forward pass; raise for throughput, lower if OOM |
| `return_logits` | `bool` | `False` | Include per-position logits in the output (large; disable to save memory) |
| `model_checkpoint` | `Literal['esm2_t6_8M_UR50D', 'esm2_t12_35M_UR50D', 'esm2_t30_150M_UR50D', 'esm2_t33_650M_UR50D', 'esm2_t36_3B_UR50D', 'esm2_t48_15B_UR50D']` | `'esm2_t33_650M_UR50D'` | ESM2 weights variant; trade off speed vs scoring fidelity |

In [17]:
# Run the scoring function
score_result = run_esm2_score(inputs, config)

Running run_esm2_score [00:00]

In [18]:
# Display output docs
display_api_reference("esm2", "output", "run_esm2_score")

# Show all scoring metrics for each input sequence
for i, score in enumerate(score_result.scores):
    print(f"Sequence {i + 1}: {inputs.sequences[i]}")
    print(f"  Log-likelihood:     {score['log_likelihood']:.3f}")
    print(f"  Avg log-likelihood: {score['avg_log_likelihood']:.3f}")
    print(f"  Perplexity:         {score['perplexity']:.3f}")

**Output** — `MaskedModelScoringOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `scores` | `list[proto_tools.tools.masked_models.shared_data_models.MaskedModelScoringMetrics]` | required | List of scoring outputs, one per input sequence |

Sequence 1: MKTAYIAKQR
  Log-likelihood:     -27.243
  Avg log-likelihood: -2.724
  Perplexity:         15.246
Sequence 2: EVQLVESGGS
  Log-likelihood:     -28.396
  Avg log-likelihood: -2.840
  Perplexity:         17.110


### `run_esm2_gradient`

The gradient tool takes a relaxed `(L, 20)` distribution over amino acids and returns the gradient of ESM2's masked pseudo-log-likelihood objective with respect to those input logits. Each AA position is masked in turn, the model predicts it from the surrounding bidirectional context, and a per-chunk backward pass accumulates the gradient. Use this as a differentiable, structure-free pLM loss inside MCMC or gradient descent over relaxed protein sequences. Set `compute_gradient=False` to get the same scalar objective without the backward pass — useful for scoring proposals.

In [19]:
from proto_tools import (
    ESM2GradientConfig,
    ESM2GradientInput,
    run_esm2_gradient,
)
from proto_tools.utils import one_hot_protein_logits


In [20]:
# Display docs
display_api_reference("esm2", "input", "run_esm2_gradient")

# Seed a relaxed distribution from a discrete sequence (sharpness=2.0
# yields a biased-but-not-saturated softmax target).
logits = one_hot_protein_logits("MKTAYIAKQR", sharpness=2.0)
inputs = ESM2GradientInput(logits=logits, temperature=0.6)


**Input** — `ESM2GradientInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `logits` | `list[list[float]]` | required | Relaxed sequence logits with shape (L, 20) in canonical amino-acid order. |
| `temperature` | `float \| None` | `None` | Softmax temperature. Applies softmax(input / T) when set. |

In [21]:
# Display config docs
display_api_reference("esm2", "config", "run_esm2_gradient")

# Configure the 650M model | see docs above for all fields
config = ESM2GradientConfig(
    model_checkpoint="esm2_t33_650M_UR50D",
    use_ste=False,        # soft blended embeddings (set True for STE)
    compute_gradient=True,
    batch_size=8,         # AA positions per forward pass
    device="cuda",        # Change to "cpu" if no GPU available
)


**Config** — `ESM2GradientConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `verbose` | `int` | `0` | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| `device` | `str` | `'cuda'` | Device to run the model on |
| `timeout` | `int` | `600` | Maximum execution time in seconds |
| `seed` | `int \| None` | `None` | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| `model_checkpoint` | `Literal['esm2_t6_8M_UR50D', 'esm2_t12_35M_UR50D', 'esm2_t30_150M_UR50D', 'esm2_t33_650M_UR50D', 'esm2_t36_3B_UR50D', 'esm2_t48_15B_UR50D']` | `'esm2_t33_650M_UR50D'` | ESM2 weights variant; trade off speed vs scoring fidelity |
| `use_ste` | `bool` | `False` | Hard one-hot forward pass with soft-probability gradients |
| `compute_gradient` | `bool` | `True` | Run backward pass and return gradient; set False for forward-only log-likelihood |
| `batch_size` | `int \| None` | `None` | AA positions per forward pass. Lower if OOM, higher for throughput |

In [22]:
# Run the gradient function
gradient_result = run_esm2_gradient(inputs, config)


Running run_esm2_gradient [00:00]

In [23]:
# Display output docs
display_api_reference("esm2", "output", "run_esm2_gradient")

# Inspect the scalar objective and gradient shape
print(f"Mean NLL:    {gradient_result.loss:.3f}")
print(f"Perplexity:  {gradient_result.metrics['perplexity']:.3f}")
print(f"Avg log-likelihood: {gradient_result.metrics['avg_log_likelihood']:.3f}")
assert gradient_result.gradient is not None  # backward pass enabled
print(f"Gradient shape: ({len(gradient_result.gradient)}, {len(gradient_result.gradient[0])})")
print(f"Gradient row 0 (first 5 cols): {[round(v, 4) for v in gradient_result.gradient[0][:5]]}")


**Output** — `ESM2GradientOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `gradient` | `list[list[float]] \| None` | `None` | Gradient w.r.t. input logits. None when compute_gradient=False. |
| `loss` | `float` | required | Scalar objective value returned by the backend for this relaxed sequence. |
| `metrics` | `dict[str, Any]` | `{}` | Auxiliary metrics reported alongside the scalar objective value. |
| `vocab` | `list[str]` | required | Amino-acid symbols defining the column ordering for both the input logits and the returned gradient. |

Mean NLL:    2.717
Perplexity:  15.134
Avg log-likelihood: -2.717
Gradient shape: (10, 20)
Gradient row 0 (first 5 cols): [0.0001, 0.0, -0.0005, -0.0005, 0.0]


## Export Results

Results from each function can be exported to standard file formats for downstream analysis. Embeddings export to CSV with one row per sequence, sampled sequences export to FASTA for compatibility with other bioinformatics tools, and scores export to JSON with full metadata.

In [24]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

embeddings_result.export("esm2_embeddings", export_path=output_dir, file_format="csv")
print(f"Embeddings exported to {output_dir / 'esm2_embeddings.csv'}")

strategy_result.export("esm2_sequences", export_path=output_dir, file_format="fasta")
print(f"Sampled sequences exported to {output_dir / 'esm2_sequences.fasta'}")

score_result.export("esm2_scores", export_path=output_dir, file_format="json")
print(f"Scores exported to {output_dir / 'esm2_scores.json'}")

Embeddings exported to example_output/esm2_embeddings.csv
Sampled sequences exported to example_output/esm2_sequences.fasta
Scores exported to example_output/esm2_scores.json
